# Embedding chunks

Goal:
- load chunked corpora (32, 128, 256)
- used a fixed embedding model (BAAI/bge-small-en-v1.5)
- convert each chunk into a dense vector using a fixed embedding model
- save embeddings for later FAISS indexing

**Important:**
- embedding model is fixed across all experiments
- embeddings are computed once and reused for all retrieval runs

In [2]:
# Load ch
import json
from pathlib import Path

data_dir = Path("/home/a/arfaoui/rag_project/data")

chunk_sizes = [32, 128, 256]

chunked_corpora = {}

for size in chunk_sizes:
    path = data_dir / f"hotpotqa_chunks_{size}.json"
    
    with open(path, "r", encoding="utf-8") as f:
        chunked_corpora[size] = json.load(f)
# print to check    
    print(f"Loaded chunk_size={size}: {len(chunked_corpora[size])} chunks")

Loaded chunk_size=32: 292199 chunks
Loaded chunk_size=128: 96686 chunks
Loaded chunk_size=256: 69845 chunks


In [3]:
from sentence_transformers import SentenceTransformer

modelName = "BAAI/bge-small-en-v1.5"

embed_model = SentenceTransformer(modelName)

print("Embedding model loaded:", modelName)

/home/a/arfaoui/miniconda3/envs/AAML/lib/python3.10/site-packages/sentence_transformers/cross_encoder/CrossEncoder.py:11: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from tqdm.autonotebook import tqdm, trange


Embedding model loaded: BAAI/bge-small-en-v1.5


In [3]:
import numpy as np

# It processes data in batches to avoid memory issues (Batch 64)

def embed_chunks(chunks, model, batch_size=64):
    """
    Convert list of chunk dicts into embedding vectors.

    Parameters:
        chunks (list): list of chunk dictionaries
        model: SentenceTransformer model
        batch_size (int): number of texts processed at once

    Returns:
        embeddings (np.ndarray): shape (num_chunks, embedding_dim)
    """
    
    # Extract only the text from each chunk
    texts = [c["text"] for c in chunks]

    # Encode texts into vector embeddings
    embeddings = model.encode(
        texts,
        batch_size=batch_size,        # process in batches
        show_progress_bar=True,       # shows progress (To recognize if it stops)
        convert_to_numpy=True         # ensures numpy array output
    )

    return embeddings

In [4]:
# Embed the full chunked corpus for each chunk size
# We keep embeddings in a dictionary so they can later be saved and indexed.

all_embeddings = {}

for size in chunk_sizes:
    print(f"\nEmbedding chunk_size={size} ...")
    
    chunks = chunked_corpora[size]
    
    # Convert all chunk texts for this chunk size into dense vectors
    embeddings = embed_chunks(
        chunks=chunks,
        model=embed_model,
        batch_size=64
    )
    
    all_embeddings[size] = embeddings
    
    # Print shape to verify number of vectors and embedding dimension
    print(f"Finished chunk_size={size}")
    print(f"Embedding matrix shape: {embeddings.shape}")


Embedding chunk_size=32 ...


Batches: 100%|██████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 4566/4566 [21:34<00:00,  3.53it/s]


Finished chunk_size=32
Embedding matrix shape: (292199, 384)

Embedding chunk_size=128 ...


Batches: 100%|██████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 1511/1511 [22:08<00:00,  1.14it/s]


Finished chunk_size=128
Embedding matrix shape: (96686, 384)

Embedding chunk_size=256 ...


Batches: 100%|██████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 1092/1092 [22:59<00:00,  1.26s/it]


Finished chunk_size=256
Embedding matrix shape: (69845, 384)


In [5]:
# Save embeddings to disk
# We use .npy format ->  fast and efficient for numpy arrays

import numpy as np

for size, embeddings in all_embeddings.items():
    path = data_dir / f"embeddings_{size}.npy"
    
    np.save(path, embeddings)
    
    print(f"Saved embeddings for chunk_size={size}: {path}")

Saved embeddings for chunk_size=32: /home/a/arfaoui/rag_project/data/embeddings_32.npy
Saved embeddings for chunk_size=128: /home/a/arfaoui/rag_project/data/embeddings_128.npy
Saved embeddings for chunk_size=256: /home/a/arfaoui/rag_project/data/embeddings_256.npy


In [6]:
# Save chunk metadata (needed to map FAISS results back to text)

for size, chunks in chunked_corpora.items():
    path = data_dir / f"chunks_metadata_{size}.json"
    
    with open(path, "w", encoding="utf-8") as f:
        json.dump(chunks, f, ensure_ascii=False)
    
    print(f"Saved metadata for chunk_size={size}: {path}")

Saved metadata for chunk_size=32: /home/a/arfaoui/rag_project/data/chunks_metadata_32.json
Saved metadata for chunk_size=128: /home/a/arfaoui/rag_project/data/chunks_metadata_128.json
Saved metadata for chunk_size=256: /home/a/arfaoui/rag_project/data/chunks_metadata_256.json


## Embedding summary


Why this matters:
- embeddings represent chunks in a semantic vector space
- FAISS will use these vectors for fast similarity search
- keeping the embedding model fixed ensures that differences in results come from chunk size and top-k.

## Observations
-> Expected observations based on the cunks observations and results 
- chunk_size=32 produced the largest embedding matrix (292,199 vectors)
- chunk_size=128 provides a balanced setting with 96,686 vectors
- chunk_size=256 results in the smallest index (69,845 vectors)